In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import shutil
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/working'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/working/__notebook__.ipynb


In [2]:
#run if needed to start from the start 
path="/kaggle/working/checkpoints"

try:
    shutil.rmtree(path)
except OSError as e:
    print(e.strerror)

No such file or directory


In [3]:
import torch 
import torch.nn as nn
import torch.optim as optim 
from torchvision import datasets , transforms , models
from torch.utils.data import DataLoader , Subset
import numpy as np
import os
import glob


In [4]:
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406] , std=[0.229, 0.224, 0.225])
])

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomPerspective(distortion_scale=.15,p=.4),
    transforms.Grayscale(num_output_channels=3),
    transforms.ColorJitter(brightness=0.4, contrast=0.4),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0,translate=(0.1,0.1),scale=(.9,1.1),shear=0),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406] , std=[0.229, 0.224, 0.225])
])



In [5]:
#dataset loading and splitting 

data_dir="/kaggle/input/hieroglyphs-dataset"
train_dataset_full=datasets.ImageFolder(root=data_dir,transform=train_transform)
val_dataset_full=datasets.ImageFolder(root=data_dir,transform=val_transform)
num_samples=len(train_dataset_full)
indices=np.random.permutation(num_samples)
split_index=int(.8*num_samples)
train_index,val_index=indices[:split_index],indices[split_index:]

train_dataset=Subset(train_dataset_full,train_index)
val_dataset=Subset(val_dataset_full,val_index)


print(f"Total images: {num_samples}")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Total images: 4031
Training samples: 3224
Validation samples: 807


In [6]:
#
batch_size=32

train_loader=DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
val_loader=DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)
print(batch_size)

32


In [7]:
num_classes=len(train_dataset_full.classes)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)


# for param in model.parameters():
#     param.requires_grad=False
# Unfreeze last 2 residual blocks + fc
for param in model.parameters():
    param.requires_grad = False

# Unfreeze layer3, layer4, and fc
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

# Use differential learning rates
optimizer = optim.Adam([
    {'params': model.layer3.parameters(), 'lr': 1e-4},
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(), 'lr': 1e-3}
])
num_features=model.fc.in_features
# model.fc=nn.Linear(num_features,num_classes)
model.fc = nn.Sequential(
    nn.Linear(num_features, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, num_classes)
)
model=model.to(device)

epochs = 100
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam([
    {'params': model.layer1.parameters(), 'lr': 1e-5},
    {'params': model.layer2.parameters(), 'lr': 1e-5},
    {'params': model.layer3.parameters(), 'lr': 1e-4},
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(), 'lr': 1e-3}
])
scheduler=optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=epochs)

print(num_classes)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 210MB/s]


171


In [8]:


epochs = 100
save_interval = 10
checkpoint_dir = '/kaggle/working/checkpoints/resnet50'

os.makedirs(checkpoint_dir, exist_ok=True)



start_epoch = 0
best_val_acc=0
best_model=None;
best_optimizer=None;


for epoch in range(start_epoch, epochs):
    
    model.train()
    running_loss=0.0
    correct_train=0
    total_train=0

    for images,labels in train_loader:
        images,labels=images.to(device) , labels.to(device)

        optimizer.zero_grad()

        outputs=model(images)
        loss=criterion(outputs,labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    epoch_train_loss = running_loss / len(train_dataset)
    epoch_train_acc = (correct_train / total_train) * 100

    # --- VALIDATION PHASE ---
    model.eval() # Set model to evaluation mode (disables dropout/batchnorm updates)
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad(): # Disable gradient calculation for validation to save memory/compute
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = (correct_val / total_val) * 100
    scheduler.step()
    
    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.2f}%")

    
    print(f"Epoch [{epoch+1}/{epochs}] completed.")
    
    # 4. Periodic Saving Logic
    # Check if the current epoch is a multiple of 10
    if epoch_val_acc>best_val_acc and epoch_val_acc > 80:
        best_val_acc=epoch_val_acc
        best_model=model
        best_optimizer=optimizer
        save_path = f"{checkpoint_dir}/model_epoch_{epoch+1}.pth"
        
        print(f"Checkpoint saved at epoch {epoch+1}: {save_path}")

torch.save({
            'model_state_dict': best_model.state_dict(),
            'optimizer_state_dict': best_optimizer.state_dict()
        }, save_path)

Epoch [1/100] Train Loss: 2.9872 Acc: 48.82% | Val Loss: 2.1110 Acc: 69.76%
Epoch [1/100] completed.
Epoch [2/100] Train Loss: 2.0318 Acc: 71.93% | Val Loss: 1.7103 Acc: 80.42%
Epoch [2/100] completed.
Checkpoint saved at epoch 2: /kaggle/working/checkpoints/resnet50/model_epoch_2.pth
Epoch [3/100] Train Loss: 1.7131 Acc: 79.09% | Val Loss: 1.5349 Acc: 84.39%
Epoch [3/100] completed.
Checkpoint saved at epoch 3: /kaggle/working/checkpoints/resnet50/model_epoch_3.pth
Epoch [4/100] Train Loss: 1.5706 Acc: 82.32% | Val Loss: 1.4575 Acc: 85.87%
Epoch [4/100] completed.
Checkpoint saved at epoch 4: /kaggle/working/checkpoints/resnet50/model_epoch_4.pth
Epoch [5/100] Train Loss: 1.4216 Acc: 86.76% | Val Loss: 1.3276 Acc: 88.85%
Epoch [5/100] completed.
Checkpoint saved at epoch 5: /kaggle/working/checkpoints/resnet50/model_epoch_5.pth
Epoch [6/100] Train Loss: 1.3461 Acc: 88.31% | Val Loss: 1.3335 Acc: 89.71%
Epoch [6/100] completed.
Checkpoint saved at epoch 6: /kaggle/working/checkpoints/r

In [9]:
# import torch
# import torchvision.models as models
# import torchvision.transforms as transforms
# from PIL import Image

# def predict_hieroglyph(image_path, model_path, num_classes=171):
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
#     # 1. Initialize Base Architecture
#     model = models.resnet50(weights=None)
    
#     # 2. Load Checkpoint FIRST to inspect the custom head dimensions
#     checkpoint = torch.load(model_path, map_location=device)
#     state_dict = checkpoint['model_state_dict']
    
#     # Extract the hidden dimension from the checkpoint's fc.0.weight tensor shape
#     # The shape is usually [out_features, in_features]
#     hidden_dim = state_dict['fc.0.weight'].shape[0]
#     num_ftrs = model.fc.in_features # 2048 for ResNet50
    
#     # 3. Reconstruct the Sequential Head
#     # Indices 1 and 2 are usually ReLU and Dropout. 
#     # Because they have no learnable parameters, their exact type won't break the state_dict loader,
#     # but using standard ReLU/Dropout is necessary for correct inference behavior.
#     model.fc = torch.nn.Sequential(
#         torch.nn.Linear(num_ftrs, hidden_dim), # fc.0
#         torch.nn.ReLU(),                       # fc.1
#         torch.nn.Dropout(p=0.5),               # fc.2
#         torch.nn.Linear(hidden_dim, num_classes) # fc.3
#     )
    
#     # 4. Load Weights
#     model.load_state_dict(state_dict) 
#     model.to(device)
#     model.eval()

#     # 5. Preprocessing
#     transform = transforms.Compose([
#         transforms.Resize((224, 224)),
#         transforms.ToTensor(),
#         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
#     ])

#     img = Image.open(image_path).convert('RGB')
#     img_tensor = transform(img).unsqueeze(0).to(device)

#     # 6. Inference
#     with torch.no_grad():
#         output = model(img_tensor)
#         _, predicted_idx = torch.max(output, 1)
    
#     return predicted_idx.item()

# # Usage
# result = predict_hieroglyph('/kaggle/input/hieroglyphs-dataset/G7/2f15b111b27ca2e9f3a725ba095cbc5f.png', '/kaggle/working/checkpoints/resnet50/model_epoch_85.pth', num_classes=171)
# print(f"Predicted Class Index: {result}")